In [72]:
# imports
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.chat_models import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import (
    FlashrankRerank,
    DocumentCompressorPipeline,
    LLMChainExtractor,
)
import chromadb

In [29]:
# load content from wiki
loader = WebBaseLoader("https://en.wikipedia.org/wiki/2024_State_of_the_Union_Address")
document = loader.load()

# split documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, length_function=len
)
documents = text_splitter.split_documents(document)

# embeddings
embeddings = OllamaEmbeddings(model="qwen3-embedding:8b")

# initialize chromadb vector store
client = chromadb.HttpClient(host="localhost", port=8000)
client.delete_collection(name="wiki_union_collection")
vector_store = Chroma(
    client=client,
    collection_name="wiki_union_collection",
    embedding_function=embeddings,
)

#save document chunks to vector store
ids = vector_store.add_documents(documents)

retriever = vector_store.as_retriever()

INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/auth/identity "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: DELETE http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/wiki_union_collection "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET http://localhost:8000/api/v2/pre-flight-checks "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f49d3986-4362-46f9-b15c-d5dfe5d11a8e/upsert "HTTP/1.1 200 OK"


In [30]:
# Query the Vector Store to validate
results = vector_store.similarity_search('Who invaded Ukraine and why is this wrong?', k=2)

for res in results:
    print(f"* {res.page_content}")

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f49d3986-4362-46f9-b15c-d5dfe5d11a8e/query "HTTP/1.1 200 OK"


* Politicians[edit]
Kathy Hochul, governor of New York, as a guest of Representative Adriano Espaillat.[42]
Garnett L. Johnson: Mayor of Augusta, Georgia, which has seen a Workforce Hub that will bring advanced manufacturing, construction, and other skilled trades to students in the area.
Ulf Kristersson: Prime Minister of Sweden, which joined NATO on the same day in response to security concerns surrounding the Russian invasion of Ukraine.
Bill Lee, governor of Tennessee, as a guest of Senator Bill Hagerty.[43]
Stephen Roe Lewis: Governor of the Gila River Indian Community who has brought innovative solutions to long-term issues confronted by the Community. His tenure has seen the completion of the first new schools on the Reservation in over 100 years and the first solar-over-canal project in the Western Hemisphere.
Olena Zelenska: First lady of Ukraine who was invited but unable to come. This came as Congress stalled aid for Ukraine in the Russo-Ukrainian war.
* Olena Zelenska: Firs

In a naive RAG flow, the user’s input query is sent directly to the vector store without any modifications. This approach can lead to performance issues, as there may be a mismatch between the query and the content needed to provide a relevant answer. In an advanced RAG flow, we address this issue with Pre-retrieval Query Rewriting.

In [46]:
def user_query_rewrite(query: str, llm: ChatOllama):
    query_rewrite_prompt = f"""
        You are a helpful assistant that takes a user's query and turns it into a short statement or paragraph so that it can be used in a semantic similarity search on a vector database to return the most similar chunks of content based on the rewritten query. Please make no comments, just return the rewritten query.

        query: {query}
        """

    retrieval_query = llm.invoke(query_rewrite_prompt)

    # Return Generated Retrieval Query
    return retrieval_query.content


# Create Document Parsing Function to String
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [77]:
# Initialize the LLM instance
llm = ChatOllama(model="gemma4:e4b", reasoning=True)

prompt_template = """
    Answer the question using ONLY the provided context. 
    If the context contains only part of the answer, provide that part and 
    state that the remaining information is not available in the text.

    Context: {context}

    Question: {query}

    Step-by-step reasoning:
    """

custom_rag_prompt = PromptTemplate.from_template(prompt_template)


In [78]:
# Flash Rerank for Post-retrieval Rerank \reorders the best chunks to the top
reranker = FlashrankRerank()

# Using an LLM Extractor for precision (actually deletes irrelevant sentences)
# This will use your Gemma model to "rewrite" the chunks to only relevant info
extractor = LLMChainExtractor.from_llm(llm)

# Combine reranker and extractor into a pipeline
# Order: Rerank first, then Extract
compressor_pipeline = DocumentCompressorPipeline(transformers=[reranker, extractor])

# Update Retriever -> Compression Retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor_pipeline, base_retriever=retriever
)

In [79]:
# Custom RAG Chain Class
class RAGChain:
    def __init__(
        self,
        llm: ChatOllama,
        retriever: ContextualCompressionRetriever,
        prompt: PromptTemplate,
    ):
        self.llm = llm
        self.retriever = retriever
        self.prompt = prompt

    def invoke(self, query: str):
        # Pre-retrieval Query Rewrite
        retrieval_query = user_query_rewrite(query, self.llm)

        print(f"Original Query: {query}\n")
        print(f"Refined Query: {retrieval_query}\n")

        # Retrieval w/ Post-retrieval Reranking
        docs = self.retriever.invoke(retrieval_query)
        context = format_docs(docs)

        # prompt template
        final_prompt = self.prompt.format(context=context, query=query)

        print(f"Final Prompt: {final_prompt}")

        # invoke LLM
        return self.llm.invoke(final_prompt)

Now that our RAG chain is ready, let’s initialize the remaining components needed for it to function: the prompt and the LLM. We will use DeepMind's `gemma4:e4b` model for the LLM instance.

In [80]:
# Initialize Custom RAG Chain
rag_chain = RAGChain(llm, compression_retriever, custom_rag_prompt)
query = "According to the context, Who invaded Ukraine and why is this wrong?"

response = rag_chain.invoke(query)

# Print Output
print(response.content)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original Query: According to the context, Who invaded Ukraine and why is this wrong?

Refined Query: What are the details concerning the invasion of Ukraine, specifically identifying the invader and explaining why the context determines that the invasion is wrong?



INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f49d3986-4362-46f9-b15c-d5dfe5d11a8e/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Final Prompt: 
    Answer the question using ONLY the provided context. 
    If the context contains only part of the answer, provide that part and 
    state that the remaining information is not available in the text.

    Context: Some of the topics mentioned by Biden included the Russo-Ukrainian war, the border crisis, the Gaza war, gun crime, rescheduling cannabis, student loan debt, medication prices, and abortion.[6]

Ulf Kristersson: Prime Minister of Sweden, which joined NATO on the same day in response to security concerns surrounding the Russian invasion of Ukraine.
Olena Zelenska: First lady of Ukraine who was invited but unable to come. This came as Congress stalled aid for Ukraine in the Russo-Ukrainian war.

    Question: According to the context, Who invaded Ukraine and why is this wrong?

    Step-by-step reasoning:
    


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


**Who invaded Ukraine:**
Russia invaded Ukraine.

**Why is this wrong:**
The provided context does not state why this information would be wrong.


In [81]:
# Invoke the chain
response = rag_chain.invoke('What is large language model?')

# Print Output
print(response.content)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Original Query: What is large language model?

Refined Query: Provide a comprehensive definition and detailed explanation of large language models (LLMs), covering their architecture, function, and applications.



INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/f49d3986-4362-46f9-b15c-d5dfe5d11a8e/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Final Prompt: 
    Answer the question using ONLY the provided context. 
    If the context contains only part of the answer, provide that part and 
    state that the remaining information is not available in the text.

    Context: 

    Question: What is large language model?

    Step-by-step reasoning:
    


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


The answer is not available in the text.
